# 🏠 House Prices - Data Preprocessing

Tento notebook obsahuje kompletní přípravu dat pro modelování.

## 📋 Obsah:

1. **Načtení dat**
2. **Log transformace SalePrice**
3. **Spojení train a test** pro konzistentní preprocessing
4. **Handling missing values**
5. **Feature Engineering**
6. **Odstranění outlierů** (pouze z train)
7. **Encoding kategorických proměnných**
8. **Fitování transformací** (StandardScaler, Imputery)
9. **Zarovnání sloupců** mezi train a test
10. **Uložení preprocessovaných dat** → `data/processed_data.pkl`

---

## ⚠️ DŮLEŽITÉ - Jak to funguje:

### **V paměti (během běhu notebooku):**
- Když spustíš buňky, proměnné (`train`, `X_train`, `y_train`) jsou v paměti Python kernelu
- Dokud je kernel spuštěný, data zůstávají v paměti
- **Když restartuješ kernel** → proměnné se vymažou z paměti

### **Na disku (trvalé):**
- Originální data (`train.csv`, `test.csv`) se **NIKDY nemění**
- Preprocessovaná data se ukládají do `data/processed_data.pkl` na konci tohoto notebooku
- Tento soubor přetrvá i po restartu kernelu

### **Správný workflow:**
1. Spusť `01_EDA.ipynb` → analyzuješ data (originální data se nemění)
2. Spusť `02_Preprocessing.ipynb` → vytvoří se `processed_data.pkl`
3. V `03_Modeling.ipynb` načti data z `processed_data.pkl` (ne z paměti!)

---

**📝 Poznámka:** Tento notebook předpokládá, že jsi už dokončil EDA v `01_EDA.ipynb`.


In [1]:
# 📦 Import knihoven
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# 📂 Import vlastních modulů
import sys
sys.path.append('..')
from src.data_loader import DataLoader
from src.feature_engineering import FeatureEngineer

# 📍 Nastavení cest
PROJECT_DIR = Path('..')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print("✅ Všechny knihovny načteny!")


✅ Všechny knihovny načteny!


## 1. Načtení dat


In [2]:
# Načtení dat
loader = DataLoader(data_dir=str(DATA_DIR))
train = loader.load_train_data()
test = loader.load_test_data()

print(f"✅ Train shape: {train.shape}")
print(f"✅ Test shape: {test.shape}")


✅ Train shape: (1460, 81)
✅ Test shape: (1459, 80)


## 2. Příprava - uložení test IDs a log transformace SalePrice


In [3]:
# Uložíme originální train a test pro preprocessing
train_original = train.copy()
test_original = test.copy()

# Uložíme test IDs (před jakoukoliv filtrací)
test_ids = test['Id'].copy()

# Log transformace SalePrice (pro lepší distribuci)
y_train = np.log1p(train['SalePrice'])
print("=" * 80)
print("📊 LOG TRANSFORMACE SALEPRICE")
print("=" * 80)
print(f"   Original skewness: {train['SalePrice'].skew():.3f}")
print(f"   After log skewness: {y_train.skew():.3f}")
print("   ✅ Transformace aplikována")

# Odstraníme SalePrice z train (budeme používat y_train)
train = train.drop('SalePrice', axis=1)

print(f"\n📊 Train shape: {train.shape}")
print(f"📊 Test shape: {test.shape}")


📊 LOG TRANSFORMACE SALEPRICE
   Original skewness: 1.883
   After log skewness: 0.121
   ✅ Transformace aplikována

📊 Train shape: (1460, 80)
📊 Test shape: (1459, 80)


## 3. Spojení train a test pro konzistentní preprocessing


In [4]:
# Spojíme train a test pro konzistentní preprocessing
n_train = train.shape[0]
all_data = pd.concat([train, test], axis=0, sort=False).reset_index(drop=True)

print("=" * 80)
print("🔗 SPOJENÍ TRAIN A TEST")
print("=" * 80)
print(f"   Combined shape: {all_data.shape}")
print(f"   Train samples: {n_train}")
print(f"   Test samples: {all_data.shape[0] - n_train}")


🔗 SPOJENÍ TRAIN A TEST
   Combined shape: (2919, 80)
   Train samples: 1460
   Test samples: 1459


## 4. Handling Missing Values


In [5]:
print("=" * 80)
print("❓ HANDLING MISSING VALUES")
print("=" * 80)

# NA znamená "None" pro tyto features (podle data_description.txt)
# Poznámka: MSSubClass je numerická proměnná (kód typu domu), ne kategorická s NA
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in none_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')
        print(f"   ✅ {col}: NA → 'None'")

# KONTEXTOVÁ LOGIKA: Numerické sloupce s kontextem
# MasVnrArea: pokud MasVnrType='None', pak 0, jinak median
if 'MasVnrArea' in all_data.columns:
    mask_none = all_data['MasVnrType'] == 'None'
    mask_missing = all_data['MasVnrArea'].isnull()
    
    # Když není masonry veneer → 0
    all_data.loc[mask_none & mask_missing, 'MasVnrArea'] = 0
    
    # Když je masonry veneer, ale hodnota chybí → median
    mask_has_masonry = ~mask_none & mask_missing
    if mask_has_masonry.sum() > 0:
        median_val = all_data[~mask_none]['MasVnrArea'].median()
        all_data.loc[mask_has_masonry, 'MasVnrArea'] = median_val
        print(f"   ✅ MasVnrArea: NA → 0 (když MasVnrType='None'), jinak median ({median_val:.0f})")
    else:
        print(f"   ✅ MasVnrArea: NA → 0 (když MasVnrType='None')")

# Basement sloupce: pokud BsmtQual='None', pak 0, jinak median
bsmt_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
             'BsmtFullBath', 'BsmtHalfBath']

for col in bsmt_cols:
    if col in all_data.columns:
        mask_none = all_data['BsmtQual'] == 'None'
        mask_missing = all_data[col].isnull()
        
        # Když není basement → 0
        all_data.loc[mask_none & mask_missing, col] = 0
        
        # Když je basement, ale hodnota chybí → median
        mask_has_basement = ~mask_none & mask_missing
        if mask_has_basement.sum() > 0:
            median_val = all_data[~mask_none][col].median()
            all_data.loc[mask_has_basement, col] = median_val
            print(f"   ✅ {col}: NA → 0 (když BsmtQual='None'), jinak median ({median_val:.0f})")
        else:
            print(f"   ✅ {col}: NA → 0 (když BsmtQual='None')")

# Garage sloupce: pokud GarageType='None', pak 0, jinak median
garage_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars']

for col in garage_cols:
    if col in all_data.columns:
        mask_none = all_data['GarageType'] == 'None'
        mask_missing = all_data[col].isnull()
        
        # Když není garage → 0
        all_data.loc[mask_none & mask_missing, col] = 0
        
        # Když je garage, ale hodnota chybí → median
        mask_has_garage = ~mask_none & mask_missing
        if mask_has_garage.sum() > 0:
            median_val = all_data[~mask_none][col].median()
            all_data.loc[mask_has_garage, col] = median_val
            print(f"   ✅ {col}: NA → 0 (když GarageType='None'), jinak median ({median_val:.0f})")
        else:
            print(f"   ✅ {col}: NA → 0 (když GarageType='None')")

# LotFrontage - nahraď median podle Neighborhood
if 'LotFrontage' in all_data.columns:
    lot_frontage_median = all_data.groupby('Neighborhood')['LotFrontage'].transform('median')
    all_data['LotFrontage'] = all_data['LotFrontage'].fillna(lot_frontage_median)
    print(f"   ✅ LotFrontage: NA → median podle Neighborhood")

# Ostatní numerické - median
numeric_cols = all_data.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if all_data[col].isnull().sum() > 0:
        median_val = all_data[col].median()
        all_data[col] = all_data[col].fillna(median_val)
        print(f"   ✅ {col}: NA → median ({median_val:.0f})")

# Ostatní kategorické - mode
categorical_cols = all_data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if all_data[col].isnull().sum() > 0:
        mode_val = all_data[col].mode()[0] if len(all_data[col].mode()) > 0 else 'None'
        all_data[col] = all_data[col].fillna(mode_val)
        print(f"   ✅ {col}: NA → mode ('{mode_val}')")

print(f"\n   ✅ Zbývající missing values: {all_data.isnull().sum().sum()}")


❓ HANDLING MISSING VALUES
   ✅ PoolQC: NA → 'None'
   ✅ MiscFeature: NA → 'None'
   ✅ Alley: NA → 'None'
   ✅ Fence: NA → 'None'
   ✅ FireplaceQu: NA → 'None'
   ✅ GarageType: NA → 'None'
   ✅ GarageFinish: NA → 'None'
   ✅ GarageQual: NA → 'None'
   ✅ GarageCond: NA → 'None'
   ✅ BsmtQual: NA → 'None'
   ✅ BsmtCond: NA → 'None'
   ✅ BsmtExposure: NA → 'None'
   ✅ BsmtFinType1: NA → 'None'
   ✅ BsmtFinType2: NA → 'None'
   ✅ MasVnrType: NA → 'None'
   ✅ MasVnrArea: NA → 0 (když MasVnrType='None')
   ✅ BsmtFinSF1: NA → 0 (když BsmtQual='None')
   ✅ BsmtFinSF2: NA → 0 (když BsmtQual='None')
   ✅ BsmtUnfSF: NA → 0 (když BsmtQual='None')
   ✅ TotalBsmtSF: NA → 0 (když BsmtQual='None')
   ✅ BsmtFullBath: NA → 0 (když BsmtQual='None')
   ✅ BsmtHalfBath: NA → 0 (když BsmtQual='None')
   ✅ GarageYrBlt: NA → 0 (když GarageType='None'), jinak median (1979)
   ✅ GarageArea: NA → 0 (když GarageType='None'), jinak median (484)
   ✅ GarageCars: NA → 0 (když GarageType='None'), jinak median (2)
   ✅ 

In [6]:
print("=" * 80)
print("🔧 FEATURE ENGINEERING")
print("=" * 80)

fe = FeatureEngineer()
all_data = fe.apply_all_features(all_data)

print(f"\n   ✅ Vytvořeno {len(fe.created_features)} nových features")
print(f"   📊 Nové features: {', '.join(fe.created_features[:10])}...")


🔧 FEATURE ENGINEERING
Created 17 new features:
['TotalSF', 'TotalBathrooms', 'HouseAge', 'RemodAge', 'IsRemodeled', 'TotalPorchSF', 'HasPool', 'HasGarage', 'HasBasement', 'HasFireplace', 'Has2ndFloor', 'HasWoodDeck', 'QualityIndex', 'GarageScore', 'KitchenScore', 'GrLivArea_x_OverallQual', 'TotalBsmtSF_x_BsmtQual']

   ✅ Vytvořeno 17 nových features
   📊 Nové features: TotalSF, TotalBathrooms, HouseAge, RemodAge, IsRemodeled, TotalPorchSF, HasPool, HasGarage, HasBasement, HasFireplace...


In [7]:
print("=" * 80)
print("🎯 ODSTRANĚNÍ OUTLIERŮ")
print("=" * 80)

# Identifikuj outliery v train části
train_data_temp = all_data[:n_train].copy()
train_data_temp['SalePrice'] = y_train.values

# Odstranění extrémních outlierů (GrLivArea > 4000 a nízká cena v log scale)
outliers_idx = train_data_temp[(train_data_temp['GrLivArea'] > 4000) & 
                               (train_data_temp['SalePrice'] < 12.5)].index

if len(outliers_idx) > 0:
    print(f"   ⚠️  Identifikováno {len(outliers_idx)} extrémních outlierů")
    print(f"   📋 Indexy: {outliers_idx.tolist()}")
    
    # Odstraníme z train data
    all_data = all_data.drop(outliers_idx).reset_index(drop=True)
    y_train = y_train.drop(outliers_idx).reset_index(drop=True)
    n_train = n_train - len(outliers_idx)
    
    print(f"   ✅ Outliery odstraněny")
    print(f"   📊 Nový train size: {n_train}")
else:
    print("   ✅ Žádné outliery k odstranění")


🎯 ODSTRANĚNÍ OUTLIERŮ
   ⚠️  Identifikováno 2 extrémních outlierů
   📋 Indexy: [523, 1298]
   ✅ Outliery odstraněny
   📊 Nový train size: 1458


## 7. Encoding kategorických proměnných


In [8]:
print("=" * 80)
print("🔤 ENCODING KATEGORICKÝCH PROMĚNNÝCH")
print("=" * 80)

# Label encoding pro ordinální proměnné (mají přirozené pořadí)
ordinal_map = {
    'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'FireplaceQu': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
}

for col, mapping in ordinal_map.items():
    if col in all_data.columns:
        all_data[col] = all_data[col].map(mapping)
        print(f"   ✅ {col}: Label encoded")

# One-hot encoding pro ostatní kategorické
categorical_features = all_data.select_dtypes(include=['object']).columns
print(f"\n   📊 Kategorické features pro one-hot encoding: {len(categorical_features)}")

if len(categorical_features) > 0:
    all_data = pd.get_dummies(all_data, columns=categorical_features, drop_first=True)
    print(f"   ✅ One-hot encoding hotový")

print(f"\n   📊 Celkový počet features po encoding: {all_data.shape[1]}")


🔤 ENCODING KATEGORICKÝCH PROMĚNNÝCH
   ✅ ExterQual: Label encoded
   ✅ ExterCond: Label encoded
   ✅ BsmtQual: Label encoded
   ✅ BsmtCond: Label encoded
   ✅ HeatingQC: Label encoded
   ✅ KitchenQual: Label encoded
   ✅ FireplaceQu: Label encoded
   ✅ GarageQual: Label encoded
   ✅ GarageCond: Label encoded

   📊 Kategorické features pro one-hot encoding: 34
   ✅ One-hot encoding hotový

   📊 Celkový počet features po encoding: 248


## 8. Fitování transformací a rozdělení na train/test

**DŮLEŽITÉ:** Fitujeme transformace (StandardScaler, Imputery) pouze na train data, pak transformujeme test data!


In [9]:
from sklearn.preprocessing import StandardScaler, RobustScaler

print("=" * 80)
print("🔄 FITOVÁNÍ TRANSFORMACÍ")
print("=" * 80)

# Rozdělíme zpět na train a test
X_train = all_data[:n_train].copy()
X_test = all_data[n_train:].copy()

print(f"\n📊 Před transformacemi:")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape: {X_test.shape}")

# Zkontrolujeme, jestli jsou všechny sloupce stejné
train_cols = set(X_train.columns)
test_cols = set(X_test.columns)
missing_in_test = train_cols - test_cols
missing_in_train = test_cols - train_cols

if missing_in_test:
    print(f"\n⚠️  Sloupce v train, které chybí v test: {missing_in_test}")
    # Přidáme chybějící sloupce do test s nulami
    for col in missing_in_test:
        X_test[col] = 0

if missing_in_train:
    print(f"\n⚠️  Sloupce v test, které chybí v train: {missing_in_train}")
    # Odstraníme z test
    X_test = X_test.drop(columns=missing_in_train)

# Zajistíme stejné pořadí sloupců
X_test = X_test[X_train.columns]

print(f"\n✅ Sloupce zarovnány")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape: {X_test.shape}")



🔄 FITOVÁNÍ TRANSFORMACÍ

📊 Před transformacemi:
   X_train shape: (1458, 248)
   X_test shape: (1459, 248)

✅ Sloupce zarovnány
   X_train shape: (1458, 248)
   X_test shape: (1459, 248)

💡 Tip: Pokud chceš použít StandardScaler nebo RobustScaler,
   fitni ho na X_train a pak transformuj X_test!


## 10. Finální kontrola a uložení preprocessovaných dat


In [11]:
print("=" * 80)
print("📊 FINÁLNÍ KONTROLA")
print("=" * 80)

print(f"\n✅ Train data:")
print(f"   Samples: {X_train.shape[0]}")
print(f"   Features: {X_train.shape[1]}")
print(f"   Missing values: {X_train.isnull().sum().sum()}")

print(f"\n✅ Test data:")
print(f"   Samples: {X_test.shape[0]}")
print(f"   Features: {X_test.shape[1]}")
print(f"   Missing values: {X_test.isnull().sum().sum()}")

print(f"\n✅ Target (log SalePrice):")
print(f"   Mean: {y_train.mean():.3f}")
print(f"   Std: {y_train.std():.3f}")
print(f"   Min: {y_train.min():.3f}")
print(f"   Max: {y_train.max():.3f}")

# Uložení preprocessovaných dat
print("\n" + "=" * 80)
print("💾 UKLÁDÁNÍ PREPROCESSOVANÝCH DAT")
print("=" * 80)

processed_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'test_ids': test_ids,
    'feature_names': X_train.columns.tolist()
}

with open(DATA_DIR / 'processed_data.pkl', 'wb') as f:
    pickle.dump(processed_data, f)

print("   ✅ Data uložena: data/processed_data.pkl")
print("\n" + "=" * 80)
print("✅ PREPROCESSING HOTOVO!")
print("=" * 80)
print("\n📝 Data jsou připravena pro modelování!")
print("💡 Můžeš pokračovat s trénováním modelů v notebooku `03_Modeling.ipynb`.")


📊 FINÁLNÍ KONTROLA

✅ Train data:
   Samples: 1458
   Features: 248
   Missing values: 118

✅ Test data:
   Samples: 1459
   Features: 248
   Missing values: 122

✅ Target (log SalePrice):
   Mean: 12.024
   Std: 0.400
   Min: 10.460
   Max: 13.534

💾 UKLÁDÁNÍ PREPROCESSOVANÝCH DAT
   ✅ Data uložena: data/processed_data.pkl

✅ PREPROCESSING HOTOVO!

📝 Data jsou připravena pro modelování!
💡 Můžeš pokračovat s trénováním modelů v notebooku `03_Modeling.ipynb`.
